# Run all notebooks

**Table of contents**<a id='toc0_'></a>    
- 1. [Settings](#toc1_)    
- 2. [Imports](#toc2_)    
- 3. [Run function](#toc3_)    
- 4. [Lists](#toc4_)    
- 5. [Run files](#toc5_)    
- 6. [Run notebooks](#toc6_)    

<!-- vscode-jupyter-toc-config
	numbering=true
	anchor=true
	flat=false
	minLevel=2
	maxLevel=6
	/vscode-jupyter-toc-config -->
<!-- THIS CELL WILL BE REPLACED ON TOC UPDATE. DO NOT WRITE YOUR TEXT IN THIS CELL -->

## 2. <a id='toc2_'></a>[Imports](#toc0_)

In [ ]:
import os
import glob
import papermill as pm
import torch
has_gpu = torch.cuda.is_available()
Ngpus = torch.cuda.device_count()

## 1. <a id='toc1_'></a>[Settings](#toc0_)

In [ ]:
run_files = False
run_notebooks = True
results = True
only_results = False
only_folders = ['0_SimpleConSavModel',
                '1_BufferStock',
                '2_Durables',
                '3_NonConvexDurables',
                '4_LargeLifeCycle'] # set to [] to run all
exclude = []

In [ ]:
notebooks = {}

notebooks['0_SimpleConSav'] = [
    ('SimpleConSavModel.ipynb','cpu'),
]

notebooks['1_BufferStock'] = [
    ('00a_Run_Tests_Main.ipynb','gpu'),
    ('00b_Run_Tests_Details.ipynb','gpu'),
    ('00c_Run_Tests_Extra.ipynb','gpu'),
    ('00d_Run_Tests_AllHyperPar.ipynb','gpu'),
    ('00e_Test_BackwardFOC.ipynb','gpu'),
    ('00f_Test_BackwardVPD.ipynb','gpu'),
    ('01_Test_DL.ipynb','gpu'),
    ('01_Test_DP.ipynb','cpu'),
    ('02_Results_DR26.ipynb','cpu'),
    ('02_Results_DR26_MoreShocks.ipynb','cpu'),
    ('02_Results_VPD.ipynb','cpu'),
    ('03_Computational_costs.ipynb','gpu'),
    #('03_Test_DDP.ipynb','gpu'),
    ('04_Find_Errors.ipynb','gpu'),
]

notebooks['2_Durables'] = [
    ('01_Run_Tests.ipynb','gpu'),
    ('01_Small_Test.ipynb','cpu'),
    ('01_Test_BackwardFOC.ipynb','gpu'),
    ('01_Test_BackwardVPD.ipynb','gpu'),    
    ('02_Results_DR26.ipynb','cpu'),
]

notebooks['3_NonConvexDurables'] = [
    ('01a_Small_Test_1D.ipynb','cpu'),
    ('01a_Small_Test_2D.ipynb','cpu'),
    ('01b_Run_DP.ipynb','cpu'),
    ('02a_Run_DL.ipynb','gpu'),
    ('02b_Backward_Improvement.ipynb','gpu'),
    ('02c_Run_DL_extra.ipynb','gpu'),
    ('02d_Run_DL_par_change.ipynb','gpu'),
    ('03_Results_DHR26.ipynb','cpu'),
]

notebooks['4_LargeLifeCycle'] = [
    ('01_Run.ipynb','gpu'),
    ('02_Backward_test.ipynb','gpu'),
    ('03_Results_DHR26.ipynb','cpu'),
]

**Notebooks not listed:**

In [ ]:
for folder,notebookspecs in notebooks.items():

    notebooklist = [notebook for notebook,_ in notebookspecs]

    print(f'### {folder} ###')
    
    notebooks_ = glob.glob(f'{folder}/*.ipynb')
    for notebook in notebooks_:
        basename = os.path.basename(notebook)
        if basename not in notebooklist:
            print(basename)

    print()

## 3. <a id='toc3_'></a>[Run function](#toc0_)

In [ ]:
def run_jupyter(filename):
    pm.execute_notebook(filename,filename)

## 5. <a id='toc5_'></a>[Run files](#toc0_)

In [ ]:
if run_files:

    if not has_gpu:

        if len(only_folders) == 0 or '1_BufferStock' in only_folders:

            os.chdir(f'{os.getcwd()}/1_BufferStock')
            os.system('python generate_DP_files_DR26.py')
            os.chdir('..')

        if len(only_folders) == 0 or '2_Durables' in only_folders:
            
            os.chdir(f'{os.getcwd()}/2_Durables')
            os.system('python generate_DP_files_DR26.py')
            os.chdir('..')

    if has_gpu:

        if len(only_folders) == 0 or '1_BufferStock' in only_folders:
        
            os.chdir(f'{os.getcwd()}/1_BufferStock')
            os.system('python generate_DL_files_DR26.py')
            os.system('python generate_DL_files_VPD.py')
            os.chdir('..')

        if len(only_folders) == 0 or '2_Durables' in only_folders:

            os.chdir(f'{os.getcwd()}/2_Durables')
            os.system('python generate_DL_files_DR26.py')
            os.chdir('..')

## 6. <a id='toc6_'></a>[Run notebooks](#toc0_)

In [ ]:
if run_notebooks:

    for folder,notebookspecs in notebooks.items():
        
        if not (len(only_folders) == 0 or folder in only_folders): continue
        
        os.chdir(f'{os.getcwd()}/{folder}')
        
        for notebook,device in notebookspecs:

            if notebook in exclude: continue
            if device == 'gpu' and not has_gpu: continue
            if device == 'cpu' and has_gpu: continue
            if 'Results' in notebook:
                if not results: continue
            else:
                if only_results: continue

            print(f'Running {folder}/{notebook} ... ',end='')
            run_jupyter(f'{notebook}')        
            print('')
            
        os.chdir('..')